In [1]:
import os
from openai import OpenAI

In [2]:
PROVIDER = "ollama"
if PROVIDER == "gemini":
  client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",   
    api_key = os.getenv("GOOGLE_API_KEY")
  )
  MODEL = "gemini-2.5-flash-lite"
elif PROVIDER == "ollama":
  client = OpenAI(
    base_url = "http://localhost:11434/v1",
    api_key = "ollama"
  )
  MODEL = "qwen3.5:397b-cloud "

In [3]:
SYSTEM_PROMPT = """
Explain me what does this code does and explain like explaining to the teen who is new: 
Tell:
1. What the code does
2. Why it is written this way
3. Suggest another/better way to do the same thing if possible
"""

In [4]:
def ask_question(question, stream=False):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        stream=stream
    )
    if stream:
        return response
    else :
        return response.choices[0].message.content

In [5]:
def main():
    print("Tech Explanation tool (type 'exit' to quit)\n")

    while True:
        question = input("Ask a question: ").strip()

        if question.lower() == "exit":
            break

        if not question:
            print("Please enter a valid question\n")
            continue

        answer = ask_question(question)
        print("\nExplanation:\n")
        use_stream = (PROVIDER == "gemini")   # Enable streaming only for Gemini

        try:
            if use_stream:
                stream = ask_question(question, stream=True)
                full_answer = ""
                for chunk in stream:
                    if chunk.choices and chunk.choices[0].delta.content:
                        text = chunk.choices[0].delta.content
                        print(text, end="", flush=True)   # Print word by word / token by token
                        full_answer += text
                print()   # New line at the end
            else:
                # Non-streaming (for Ollama)
                answer = ask_question(question, stream=False)
                print(answer)
            print(answer)
            print("\n" + "=" * 50 + "\n")
        except Exception as e:
            print(f"Error occurred: {e}\n")

In [6]:
if __name__ == "__main__":
    main()

Tech Explanation tool (type 'exit' to quit)


Explanation:

Since Docker isn't a specific piece of code but rather a tool/platform, I'll adapt the structure you asked for to explain the concept clearly!

Here is the breakdown of Docker, explained simply:

### 1. What Docker does
Imagine you built a really cool Lego castle. You want to show it to your friend, but when they try to build it using their own Lego bricks, it falls apart because their bricks are a slightly different shape or they're missing a specific piece.

**Docker solves this.** It puts your application (your code) and everything it needs to run (settings, tools, libraries) into a standardized box called a **"Container."**

*   **In simple terms:** Docker packages your software so it runs exactly the same way on your laptop, your friend's computer, or a massive server in the cloud. It stops the "It works on my machine!" problem.

### 2. Why it is written (designed) this way
Before Docker, if you wanted to run an app, you 